Caso 1

In [1]:
# Funciones auxiliares — Problema 1: Regresión (Housing)
# Son las mismas que usamos en los laboratorios de regresión lineal y árboles (caso MPG).
# Ejecuta esta celda una vez; después sólo llama a las funciones.
import numpy as np
import pandas as pd
import plotly.express as px
from sklearn.metrics import (mean_absolute_error, mean_absolute_percentage_error,
                             mean_squared_error, r2_score)


def plot_frequencies(df, columns, top_n=None):
    """Gráfica de barras con la frecuencia de cada categoría (top_n limita a las más comunes)."""
    for col in columns:
        freq = df[col].value_counts()
        if top_n:
            freq = freq.head(top_n)
        freq_df = freq.rename_axis(col).reset_index(name='Frecuencia')

        title = f'Frecuencias de {col}'
        if top_n and df[col].nunique() > top_n:
            title += f' (top {top_n})'

        fig = px.bar(freq_df, x=col, y='Frecuencia', title=title)
        fig.update_layout(xaxis={'categoryorder': 'total descending'})
        fig.show()


def evaluar_regresion(nombre, y_true, y_pred):
    """Imprime y regresa R², RMSE, MAE y MAPE (%) de un modelo de regresión."""
    metricas = {
        'modelo': nombre,
        'R²': round(r2_score(y_true, y_pred), 4),
        'RMSE': round(np.sqrt(mean_squared_error(y_true, y_pred)), 4),
        'MAE': round(mean_absolute_error(y_true, y_pred), 3),
        'MAPE (%)': round(mean_absolute_percentage_error(y_true, y_pred) * 100, 2),
    }
    print(f"{nombre}: R² {metricas['R²']} | RMSE {metricas['RMSE']} | "
          f"MAE {metricas['MAE']} | MAPE {metricas['MAPE (%)']}%")
    return metricas


def plot_importancias(modelo, columnas, top_n=None):
    """Importancia de cada variable en un modelo basado en árboles."""
    imp = pd.Series(modelo.feature_importances_, index=columnas).sort_values(ascending=True)
    if top_n:
        imp = imp.tail(top_n)

    fig = px.bar(
        x=imp.values, y=imp.index, orientation='h', text_auto='.3f',
        labels={'x': 'Importancia (reducción de impureza)', 'y': ''},
        title='Importancia de variables', template='plotly_white'
    )
    fig.update_layout(width=800, height=450)
    fig.show()
    return imp.sort_values(ascending=False)

In [2]:
df =pd.read_csv("https://raw.githubusercontent.com/jorgermzg15/data-files/refs/heads/main/Housing.csv")

In [3]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 545 entries, 0 to 544
Data columns (total 13 columns):
 #   Column            Non-Null Count  Dtype
---  ------            --------------  -----
 0   price             545 non-null    int64
 1   area              545 non-null    int64
 2   bedrooms          545 non-null    int64
 3   bathrooms         545 non-null    int64
 4   stories           545 non-null    int64
 5   mainroad          545 non-null    str  
 6   guestroom         545 non-null    str  
 7   basement          545 non-null    str  
 8   hotwaterheating   545 non-null    str  
 9   airconditioning   545 non-null    str  
 10  parking           545 non-null    int64
 11  prefarea          545 non-null    str  
 12  furnishingstatus  545 non-null    str  
dtypes: int64(6), str(7)
memory usage: 55.5 KB


In [4]:
df.dtypes.value_counts()

str      7
int64    6
Name: count, dtype: int64

In [5]:
df['furnishingstatus'].head()

0         furnished
1         furnished
2    semi-furnished
3         furnished
4         furnished
Name: furnishingstatus, dtype: str

In [6]:
from sklearn.model_selection import train_test_split

In [7]:
# Identifica la variable objetivo y separa las variables predictoras.
columnas_binarias = [
    'mainroad',
    'guestroom',
    'basement',
    'hotwaterheating',
    'airconditioning',
    'prefarea'
]

for columna in columnas_binarias:
    df[columna] = df[columna].map({'yes': 1, 'no': 0})

df = pd.get_dummies(
    df,
    columns=['furnishingstatus'],
    drop_first=True,
    dtype=int
)

X = df.drop(columns='price')
y = df['price']

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42
)

In [8]:
print(X_train.shape)
print(X_test.shape)
print(y_train.shape)
print(y_test.shape)

(436, 13)
(109, 13)
(436,)
(109,)


In [9]:
evaluar_regresion("Modelo Base", y_test, [y_train.mean()] * len(y_test))

Modelo Base: R² -0.0179 | RMSE 2268298.2282 | MAE 1748624.607 | MAPE 39.89%


{'modelo': 'Modelo Base',
 'R²': -0.0179,
 'RMSE': np.float64(2268298.2282),
 'MAE': 1748624.607,
 'MAPE (%)': 39.89}

In [10]:
from sklearn.tree import DecisionTreeRegressor

In [11]:
model = DecisionTreeRegressor(random_state=42)
model.fit(X_train, y_train)

y_pred = model.predict(X_test)
evaluar_regresion("Árbol de decisión", y_test, y_pred)



Árbol de decisión: R² 0.4771 | RMSE 1625669.9043 | MAE 1195266.055 | MAPE 24.93%


{'modelo': 'Árbol de decisión',
 'R²': 0.4771,
 'RMSE': np.float64(1625669.9043),
 'MAE': 1195266.055,
 'MAPE (%)': 24.93}

Caso 2

In [12]:
# Funciones auxiliares — Problema 2: Clasificación (UCI Credit Card)
# Son las mismas que usamos en los laboratorios de regresión logística y árboles (caso Titanic).
# Ejecuta esta celda una vez; después sólo llama a las funciones.
import numpy as np
import pandas as pd
import plotly.express as px
from sklearn.metrics import (accuracy_score, confusion_matrix, f1_score,
                             precision_score, recall_score)
from sklearn.preprocessing import StandardScaler


def plot_frequencies(df, columns, top_n=None):
    """Gráfica de barras con la frecuencia de cada categoría (top_n limita a las más comunes)."""
    for col in columns:
        freq = df[col].value_counts()
        if top_n:
            freq = freq.head(top_n)
        freq_df = freq.rename_axis(col).reset_index(name='Frecuencia')

        title = f'Frecuencias de {col}'
        if top_n and df[col].nunique() > top_n:
            title += f' (top {top_n})'

        fig = px.bar(freq_df, x=col, y='Frecuencia', title=title)
        fig.update_layout(xaxis={'categoryorder': 'total descending'})
        fig.show()


def escalar(X_train, X_test):
    """Estandariza (media 0, desviación 1) ajustando sólo con train. Regresa ambos DataFrames."""
    escalador = StandardScaler().fit(X_train)
    X_train_s = pd.DataFrame(escalador.transform(X_train), columns=X_train.columns, index=X_train.index)
    X_test_s = pd.DataFrame(escalador.transform(X_test), columns=X_test.columns, index=X_test.index)
    return X_train_s, X_test_s


def evaluar_clasificador(nombre, y_true, y_pred):
    """Imprime y regresa exactitud (accuracy), precisión, recall y F1.

    Con dos clases, precisión, recall y F1 se calculan para la clase positiva (1).
    Con más de dos clases son el promedio simple de las clases (average='macro').
    """
    binario = len(np.unique(np.concatenate([np.asarray(y_true), np.asarray(y_pred)]))) <= 2
    promedio = 'binary' if binario else 'macro'
    metricas = {
        'modelo': nombre,
        'exactitud': round(accuracy_score(y_true, y_pred), 4),
        'precisión': round(precision_score(y_true, y_pred, average=promedio, zero_division=0), 4),
        'recall': round(recall_score(y_true, y_pred, average=promedio, zero_division=0), 4),
        'F1': round(f1_score(y_true, y_pred, average=promedio, zero_division=0), 4),
    }
    print(f"{nombre}: exactitud {metricas['exactitud']} | precisión {metricas['precisión']} "
          f"| recall {metricas['recall']} | F1 {metricas['F1']}")
    return metricas


def plot_matriz_confusion(y_true, y_pred, clases=None, etiquetas=None, title='Matriz de confusión'):
    """Aciertos y errores del clasificador: la diagonal son los aciertos.

    clases: los valores de la variable objetivo en el orden en que quieres verlos, p. ej. [0, 1].
            Si no las pasas, se ordenan de menor a mayor.
    etiquetas: nombres a mostrar para esas clases (opcional), p. ej. ['Pagó', 'Incumplió'].
    """
    if clases is None:
        clases = sorted(set(np.asarray(y_true)) | set(np.asarray(y_pred)))
    clases = list(clases)
    etiquetas = list(etiquetas) if etiquetas is not None else clases

    cm = pd.DataFrame(
        confusion_matrix(y_true, y_pred, labels=clases),
        index=[f'Real: {e}' for e in etiquetas],
        columns=[f'Predicho: {e}' for e in etiquetas]
    )

    fig = px.imshow(cm, text_auto=True, color_continuous_scale='Blues', title=title)
    fig.update_layout(width=650, height=450)
    fig.show()
    return cm

In [13]:
df1 = pd.read_csv("https://raw.githubusercontent.com/jorgermzg15/data-files/refs/heads/main/UCI_Credit_Card.csv")

In [14]:
#change column name  ‘default.payment.next.month' to ‘IsDefaulter’ using rename pandas
df1.rename(columns={'default.payment.next.month': 'IsDefaulter'}, inplace=True)

In [15]:
df1['IsDefaulter'].value_counts()

IsDefaulter
0    23364
1     6636
Name: count, dtype: int64

In [16]:
#group values 0, 5 and 6 in column EDUCATION into value 4
df1['EDUCATION'] = df1['EDUCATION'].replace({0: 4, 5: 4, 6: 4})

In [17]:
df1['EDUCATION'].value_counts()

EDUCATION
2    14030
1    10585
3     4917
4      468
Name: count, dtype: int64

In [18]:
df1['MARRIAGE'] = df1['MARRIAGE'].replace({0: 3})

In [19]:
df1['MARRIAGE'].value_counts()

MARRIAGE
2    15964
1    13659
3      377
Name: count, dtype: int64

In [20]:
#mean age of IsDefaulter
df1.groupby('IsDefaulter')['AGE'].mean()

IsDefaulter
0    35.417266
1    35.725738
Name: AGE, dtype: float64

In [21]:
#apply one hot enconding to EDUCATION and MARRIAGE columns
df1 = pd.get_dummies(df1, columns=['EDUCATION', 'MARRIAGE'], drop_first=True)

In [22]:
dummies_education = [c for c in df1.columns if c.startswith('EDUCATION_')]
dummies_marriage = [c for c in df1.columns if c.startswith('MARRIAGE_')]

print("Educación:", len(dummies_education))
print("Estado civil:", len(dummies_marriage))
print("Total:", len(dummies_education) + len(dummies_marriage))

Educación: 3
Estado civil: 2
Total: 5


In [23]:
df1 = pd.get_dummies(df1, columns=['PAY_0', 'PAY_2', 'PAY_3', 'PAY_4', 'PAY_5', 'PAY_6'], drop_first=True)

In [24]:
dummies_pay0 = [c for c in df1.columns if c.startswith('PAY_0_')]
dummies_pay2 = [c for c in df1.columns if c.startswith('PAY_2_')]
dummies_pay3 = [c for c in df1.columns if c.startswith('PAY_3_')]  
dummies_pay4 = [c for c in df1.columns if c.startswith('PAY_4_')]
dummies_pay5 = [c for c in df1.columns if c.startswith('PAY_5_')]
dummies_pay6 = [c for c in df1.columns if c.startswith('PAY_6_')]
print("PAY_0:", len(dummies_pay0))
print("PAY_2:", len(dummies_pay2))
print("PAY_3:", len(dummies_pay3))
print("PAY_4:", len(dummies_pay4))
print("PAY_5:", len(dummies_pay5))
print("PAY_6:", len(dummies_pay6))


PAY_0: 10
PAY_2: 10
PAY_3: 10
PAY_4: 10
PAY_5: 9
PAY_6: 9


In [25]:
#drop ID column from df1
df1 = df1.drop(columns=['ID'])

In [26]:
df1.info()

<class 'pandas.DataFrame'>
RangeIndex: 30000 entries, 0 to 29999
Data columns (total 79 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   LIMIT_BAL    30000 non-null  float64
 1   SEX          30000 non-null  int64  
 2   AGE          30000 non-null  int64  
 3   BILL_AMT1    30000 non-null  float64
 4   BILL_AMT2    30000 non-null  float64
 5   BILL_AMT3    30000 non-null  float64
 6   BILL_AMT4    30000 non-null  float64
 7   BILL_AMT5    30000 non-null  float64
 8   BILL_AMT6    30000 non-null  float64
 9   PAY_AMT1     30000 non-null  float64
 10  PAY_AMT2     30000 non-null  float64
 11  PAY_AMT3     30000 non-null  float64
 12  PAY_AMT4     30000 non-null  float64
 13  PAY_AMT5     30000 non-null  float64
 14  PAY_AMT6     30000 non-null  float64
 15  IsDefaulter  30000 non-null  int64  
 16  EDUCATION_2  30000 non-null  bool   
 17  EDUCATION_3  30000 non-null  bool   
 18  EDUCATION_4  30000 non-null  bool   
 19  MARRIAGE_2   30

In [27]:
X = df1.drop(columns="IsDefaulter")
y = df1["IsDefaulter"]

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.33,
    random_state=42
)

In [28]:
len(X_train)

20100

In [29]:
from sklearn.linear_model import LogisticRegression

In [30]:
#Use standard scaler for the data
Scaler = StandardScaler()
X_train_scaled = Scaler.fit_transform(X_train)
X_test_scaled = Scaler.transform(X_test)
#Adjust a logistic regression with the requested parameters on the balanced, transformed, and scaled training set.
logistic_model = LogisticRegression(C=10, penalty='l2', random_state=42, class_weight="balanced", max_iter=2000)
logistic_model.fit(X_train_scaled, y_train)
y_pred = logistic_model.predict(X_test_scaled)

#accuracy en test
evaluar_clasificador("Regresión logística", y_test, y_pred)

/workspaces/aprendizaje-no-supervisado-student/.venv/lib/python3.14/site-packages/sklearn/linear_model/_logistic.py:1381: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning, leave 'penalty' set to its default value and use 'l1_ratio' or 'C' instead. Use l1_ratio=0 instead of penalty='l2', l1_ratio=1 instead of penalty='l1', l1_ratio set to a float between 0 and 1 instead of penalty='elasticnet', and C=np.inf instead of penalty=None.
  warnings.warn(


Regresión logística: exactitud 0.7743 | precisión 0.4853 | recall 0.5806 | F1 0.5287


{'modelo': 'Regresión logística',
 'exactitud': 0.7743,
 'precisión': 0.4853,
 'recall': 0.5806,
 'F1': 0.5287}

In [31]:
plot_matriz_confusion(y_test, y_pred, clases=[0, 1], etiquetas=['No incumplió', 'Incumplió'], title='Matriz de confusión: Regresión logística')

,Predicho: No incumplió,Predicho: Incumplió
Real: No incumplió,6413,1329
Real: Incumplió,905,1253


In [32]:
evaluar_clasificador("Árbol de decisión", y_test, y_pred)

Árbol de decisión: exactitud 0.7743 | precisión 0.4853 | recall 0.5806 | F1 0.5287


{'modelo': 'Árbol de decisión',
 'exactitud': 0.7743,
 'precisión': 0.4853,
 'recall': 0.5806,
 'F1': 0.5287}

In [33]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix

# Carga y preparación completa de los datos de clasificación.
df1_arbol = pd.read_csv("https://raw.githubusercontent.com/jorgermzg15/data-files/refs/heads/main/UCI_Credit_Card.csv")
df1_arbol = df1_arbol.rename(columns={'default.payment.next.month': 'IsDefaulter'})
df1_arbol['EDUCATION'] = df1_arbol['EDUCATION'].replace({0: 4, 5: 4, 6: 4})
df1_arbol['MARRIAGE'] = df1_arbol['MARRIAGE'].replace({0: 3})

df1_arbol = pd.get_dummies(
    df1_arbol,
    columns=['EDUCATION', 'MARRIAGE', 'PAY_0', 'PAY_2', 'PAY_3', 'PAY_4', 'PAY_5', 'PAY_6'],
    drop_first=True,
    dtype=int
)
df1_arbol = df1_arbol.drop(columns=['ID'])

X_arbol = df1_arbol.drop(columns='IsDefaulter')
y_arbol = df1_arbol['IsDefaulter']

X_train_arbol, X_test_arbol, y_train_arbol, y_test_arbol = train_test_split(
    X_arbol,
    y_arbol,
    test_size=0.33,
    random_state=42
)

modelo_arbol = DecisionTreeClassifier(
    max_depth=20,
    min_samples_split=0.1,
    random_state=42,
    class_weight="balanced"
)

modelo_arbol.fit(X_train_arbol, y_train_arbol)
y_pred_arbol = modelo_arbol.predict(X_test_arbol)

print("Accuracy:", round(accuracy_score(y_test_arbol, y_pred_arbol), 4))
print("Precision:", round(precision_score(y_test_arbol, y_pred_arbol, zero_division=0), 4))
print("Recall:", round(recall_score(y_test_arbol, y_pred_arbol, zero_division=0), 4))
print("F1:", round(f1_score(y_test_arbol, y_pred_arbol, zero_division=0), 4))
print("Matriz de confusión:")
print(confusion_matrix(y_test_arbol, y_pred_arbol, labels=[0, 1]))

Accuracy: 0.7408
Precision: 0.4322
Recall: 0.6024
F1: 0.5033
Matriz de confusión:
[[6034 1708]
 [ 858 1300]]


In [34]:
plot_matriz_confusion(y_test_arbol, y_pred_arbol, clases=[0, 1], etiquetas=['No incumplió', 'Incumplió'], title='Matriz de confusión: Regresión logística')

,Predicho: No incumplió,Predicho: Incumplió
Real: No incumplió,6034,1708
Real: Incumplió,858,1300


Caso 3

In [35]:
# Funciones auxiliares — Problema 3: Clustering (College Data)
# Son las mismas que usamos en los laboratorios de K-Means, DBSCAN y PCA.
# Ejecuta esta celda una vez; después sólo llama a las funciones.
#   Nuevo: evaluar_clustering ahora reporta DOS silhouettes:
#        'silhouette'           → como la calcula sklearn: el ruido (-1) cuenta como un grupo más.
#        'silhouette_sin_ruido' → como en clase: sólo los puntos que quedaron en un cluster.
#   Nuevo: tabla_metricas_k usa n_init='auto' (el valor por defecto de KMeans) en vez de n_init=10.
import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from sklearn.cluster import DBSCAN, KMeans
from sklearn.metrics import silhouette_score


def evaluar_clustering(nombre, X, labels):
    """Número de clusters (sin contar el ruido), % de ruido y silhouette de un agrupamiento."""
    X = np.asarray(X)
    labels = np.asarray(labels)
    mask = labels != -1
    n_clusters = len(np.unique(labels[mask]))
    fila = {
        'modelo': nombre,
        'n_clusters': n_clusters,
        'pct_ruido': float(round(100 * (~mask).mean(), 1)),
    }
    # sklearn: todas las etiquetas, el ruido incluido como un grupo (necesita al menos 2 etiquetas)
    fila['silhouette'] = (float(round(silhouette_score(X, labels), 3))
                          if len(np.unique(labels)) >= 2 else np.nan)
    # Sin ruido: sólo los puntos asignados a un cluster (necesita al menos 2 clusters)
    fila['silhouette_sin_ruido'] = (float(round(silhouette_score(X[mask], labels[mask]), 3))
                                    if n_clusters >= 2 else np.nan)
    return fila


def tabla_metricas_k(X, ks, random_state=42):
    """Ajusta K-Means para cada k y regresa una tabla con la inercia (para el codo) y la silhouette."""
    filas = []
    for k in ks:
        modelo = KMeans(n_clusters=k, n_init='auto', random_state=random_state).fit(X)
        fila = evaluar_clustering(f'k={k}', X, modelo.labels_)
        filas.append({'k': k, 'inercia': float(round(modelo.inertia_, 1)), 'silhouette': fila['silhouette']})
    return pd.DataFrame(filas)


def grilla_dbscan(X, eps_values, min_samples_values):
    """Ajusta DBSCAN para cada combinación de eps y min_samples.
    Regresa una tabla con el número de clusters, % de ruido y ambas silhouettes."""
    filas = []
    for eps in eps_values:
        for ms in min_samples_values:
            labels = DBSCAN(eps=eps, min_samples=ms).fit_predict(X)
            fila = evaluar_clustering(f'eps={eps}, ms={ms}', X, labels)
            filas.append({'eps': eps, 'min_samples': ms, **fila})
    return pd.DataFrame(filas).drop(columns='modelo')


def perfil_clusters(df, labels, columnas):
    """Promedio de cada variable por cluster, con tamaño y porcentaje. Sirve para ponerle nombre a cada grupo."""
    datos = df[columnas].copy()
    datos['cluster'] = np.asarray(labels)
    perfil = datos.groupby('cluster')[columnas].mean().round(1)
    perfil.insert(0, 'n', datos['cluster'].value_counts().sort_index())
    perfil.insert(1, 'pct', (100 * perfil['n'] / perfil['n'].sum()).round(1))
    perfil.index = ['Ruido' if c == -1 else f'Cluster {c}' for c in perfil.index]
    return perfil


def plot_codo(ks, inercias):
    """Método del codo: inercia (suma de distancias² al centroide) para cada k."""
    fig = px.line(
        x=list(ks),
        y=list(inercias),
        markers=True,
        labels={'x': 'Número de clusters (k)', 'y': 'Inercia'},
        title='Método del codo'
    )
    fig.update_layout(width=750, height=450)
    fig.show()


def plot_silueta_k(tabla):
    """Silhouette promedio para cada k (tabla de tabla_metricas_k). Más alto es mejor."""
    fig = px.line(
        tabla,
        x='k',
        y='silhouette',
        markers=True,
        labels={'k': 'Número de clusters (k)', 'silhouette': 'Silhouette promedio'},
        title='Silhouette según k (↑ mejor)'
    )
    fig.update_xaxes(dtick=1)
    fig.update_layout(width=750, height=450)
    fig.show()


def plot_k_distancia(X, k, eps=None):
    """Distancia que necesita cada punto para tener k vecinos (contándose a sí mismo), ordenada.
    Usa k = min_samples. El 'codo' sugiere un buen eps; eps (opcional) dibuja una línea de referencia."""
    from sklearn.neighbors import NearestNeighbors

    vecinos = NearestNeighbors(n_neighbors=k).fit(X)
    distancias, _ = vecinos.kneighbors(X)
    d_k = np.sort(distancias[:, -1])

    fig = px.line(
        x=np.arange(len(d_k)),
        y=d_k,
        labels={'x': 'Puntos ordenados por distancia', 'y': f'Distancia para tener {k} vecinos (incluido él)'},
        title=f'Gráfica de k-distancia (k = {k})'
    )
    if eps is not None:
        fig.add_hline(y=eps, line_dash='dash', line_color='red',
                      annotation_text=f'eps = {eps}', annotation_position='top left')
    fig.update_layout(width=750, height=450)
    fig.show()


def plot_varianza_explicada(pca):
    """Scree plot: varianza explicada por componente (barras) y acumulada (línea)."""
    ratio = pca.explained_variance_ratio_
    n = np.arange(1, len(ratio) + 1)
    fig = go.Figure()
    fig.add_trace(go.Bar(x=n, y=ratio, name='Por componente'))
    fig.add_trace(go.Scatter(x=n, y=np.cumsum(ratio), mode='lines+markers', name='Acumulada'))
    for umbral in (0.80, 0.95):
        fig.add_hline(y=umbral, line_dash='dot', line_color='gray',
                      annotation_text=f'{umbral:.0%}', annotation_position='right')
    fig.update_layout(
        title='Varianza explicada por componente',
        xaxis={'title': 'Componente principal', 'dtick': 1 if len(ratio) <= 30 else None},
        yaxis={'title': 'Proporción de varianza', 'tickformat': '.0%', 'range': [0, 1.05]},
        width=800,
        height=450
    )
    fig.show()

In [36]:
df2 = pd.read_csv("https://raw.githubusercontent.com/jorgermzg15/data-files/refs/heads/main/College_Data.csv")

In [37]:
df2 = df2.drop(columns=['College'])

In [38]:
df2['Private'] = df2['Private'].map({'Yes': 1, 'No': 0})

In [39]:
df2.info()

<class 'pandas.DataFrame'>
RangeIndex: 777 entries, 0 to 776
Data columns (total 18 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   Private      777 non-null    int64  
 1   Apps         777 non-null    int64  
 2   Accept       777 non-null    int64  
 3   Enroll       777 non-null    int64  
 4   Top10perc    777 non-null    int64  
 5   Top25perc    777 non-null    int64  
 6   F.Undergrad  777 non-null    int64  
 7   P.Undergrad  777 non-null    int64  
 8   Outstate     777 non-null    int64  
 9   Room.Board   777 non-null    int64  
 10  Books        777 non-null    int64  
 11  Personal     777 non-null    int64  
 12  PhD          777 non-null    int64  
 13  Terminal     777 non-null    int64  
 14  S.F.Ratio    777 non-null    float64
 15  perc.alumni  777 non-null    int64  
 16  Expend       777 non-null    int64  
 17  Grad.Rate    777 non-null    int64  
dtypes: float64(1), int64(17)
memory usage: 109.4 KB


In [40]:
tabla_metricas_k((df2), ks=range(2, 3), random_state=42)

,k,inercia,silhouette
0,2,4.926657e+10,0.537


In [41]:
from sklearn.cluster import DBSCAN

escalador = StandardScaler()
df2_scaled = escalador.fit_transform(df2)

labels_dbscan = DBSCAN(eps=2.0, min_samples=5).fit_predict(df2_scaled)

print(evaluar_clustering(
    "DBSCAN eps=2.0, ms=5",
    df2_scaled,
    labels_dbscan
))

{'modelo': 'DBSCAN eps=2.0, ms=5', 'n_clusters': 3, 'pct_ruido': 37.2, 'silhouette': 0.041, 'silhouette_sin_ruido': 0.256}


In [42]:
from sklearn.decomposition import PCA

In [43]:
# Aplica PCA con n_components=0.8 y random_state=42 a los datos escalados (tienes que crear X_pca). Después ajusta K-Means con n_clusters=3 y random_state=42 sobre X_pca.


pca = PCA(n_components=0.8, random_state=42)
X_pca = pca.fit_transform(df2_scaled)

kmeans = KMeans(n_clusters=3, random_state=42)
labels_kmeans = kmeans.fit_predict(X_pca)

print(evaluar_clustering(
    "K-Means k=3",
    X_pca,
    labels_kmeans
))

{'modelo': 'K-Means k=3', 'n_clusters': 3, 'pct_ruido': 0.0, 'silhouette': 0.292, 'silhouette_sin_ruido': 0.292}
